# SCI Student

## Import Library

In [2]:
import os
import pandas as pd
import numpy as np
import glob
import warnings
import re
import datetime

In [3]:
from pathlib import Path

## Reading File

In [ ]:
file_name = r'' # Update the input directory path here
df = pd.read_excel(file_name, sheet_name='All courses for SCI student')

# --- NEW: Save a copy of the original data for the second sheet ---
df_records = df.copy() 

print(f"Total rows loaded: {len(df)}")
df.columns

Total rows loaded: 15000


Index(['Student ID', 'First name', 'Last name', 'Major', 'Dept',
       'Course Number', 'Course Type', 'Section', 'Credits', 'Grade',
       'Enroll Status', 'PT FT Status', 'Term'],
      dtype='object')

## Processing 

In [5]:
df['Dept'] = df['Dept'].astype(str).str.strip().str.upper()
df['Course Number'] = df['Course Number'].astype(str).str.strip()
df['Term'] = df['Term'].astype(str).str.strip().str.upper()
df['Grade'] = df['Grade'].astype(str).str.strip().str.upper()

## Processing 
# Strip whitespace and force course numbers to strings to ensure consistent matching
df['Dept'] = df['Dept'].str.strip()
df['Course Number'] = df['Course Number'].astype(str).str.strip()

# --- Step 1: Map Grades to Numbers (Challenge 2) ---
# Map Grades (Notice 'W' is left out, it will become NaN and ignored in the GPA average)
grade_mapping = {
    'A': 4.0, 'A-': 3.7, 'B+': 3.3, 'B': 3.0, 'B-': 2.7,
    'C+': 2.3, 'C': 2.0, 'C-': 1.7, 'D+': 1.3, 'D': 1.0, 'F': 0.0
}
df['Num_Grade'] = df['Grade'].map(grade_mapping)

# --- Step 2: Make Terms Chronological (Challenge 4) ---
def term_to_numeric(term):
    if pd.isna(term) or term == 'NAN': return np.nan
    season = term[:2]
    try:
        year = int(term[2:4])
    except ValueError:
        return np.nan # Failsafe if the term format is weird
    season_weights = {'SP': 0.1, 'SU': 0.2, 'FA': 0.3}
    return year + season_weights.get(season, 0)

df['Term_Num'] = df['Term'].apply(term_to_numeric)

# Filter for the specific course
sec110_df = df[(df['Dept'] == 'SEC') & (df['Course Number'] == '110')]
print(f"Total SEC 110 records found: {len(sec110_df)}") 

if len(sec110_df) == 0:
    print("WARNING: No students found who took SEC 110. Check your Dept and Course Number values.")

# Get target terms and map back to main dataframe
target_terms = sec110_df.groupby('Student ID')['Term_Num'].min().to_dict()
df['Target_Term_Num'] = df['Student ID'].map(target_terms)

# Drop students who never took it
df = df.dropna(subset=['Target_Term_Num'])
print(f"Total rows after keeping only SEC 110 students: {len(df)}")

# Tag Timeline
conditions = [
    df['Term_Num'] < df['Target_Term_Num'],
    df['Term_Num'] == df['Target_Term_Num'],
    df['Term_Num'] > df['Target_Term_Num']
]
choices = ['Pre', 'Target', 'Post']
df['Timeline'] = np.select(conditions, choices, default='Unknown')

# Calculate averages (pandas .mean() automatically ignores the NaNs caused by 'W' grades)
student_gpas = df.groupby(['Student ID', 'Timeline'])['Num_Grade'].mean().unstack()

# Apply Difference Logic
def calculate_performance_shift(row):
    if pd.notna(row.get('Pre')) and pd.notna(row.get('Post')):
        return row['Post'] - row['Pre']
    elif pd.isna(row.get('Pre')) and pd.notna(row.get('Post')):
        return row['Post'] - row.get('Target', 0) 
    elif pd.notna(row.get('Pre')) and pd.isna(row.get('Post')):
        return row.get('Target', 0) - row['Pre']
    else:
        return np.nan

student_gpas['GPA_Difference'] = student_gpas.apply(calculate_performance_shift, axis=1)

print("\n--- FINAL OUTPUT ---")
print(student_gpas)

final_output_df = student_gpas.reset_index()

# Remove the confusing 'Timeline' name from the columns axis caused by unstacking
final_output_df.columns.name = None 

print("\n--- FINAL OUTPUT WITH STUDENT ID ---")
print(final_output_df.head(15)) # Prints the first 15 rows

Total SEC 110 records found: 2526
Total rows after keeping only SEC 110 students: 12614

--- FINAL OUTPUT ---
Timeline        Post       Pre    Target  GPA_Difference
Student ID                                              
10101614    2.650000       NaN  1.333333        1.316667
10114861    2.328571       NaN  3.700000       -1.371429
10143308    1.633333       NaN  1.500000        0.133333
10431919    2.185714       NaN  0.675000        1.510714
10512202    2.600000  1.500000  2.333333        1.100000
...              ...       ...       ...             ...
99370962    1.600000  2.622222  3.300000       -1.022222
99384351    2.285714       NaN  1.300000        0.985714
99426504    0.000000  1.636364  1.500000       -1.636364
99447374    1.757143  2.000000  1.900000       -0.242857
99920773    1.000000  1.757143  1.000000       -0.757143

[1215 rows x 4 columns]

--- FINAL OUTPUT WITH STUDENT ID ---
    Student ID      Post       Pre    Target  GPA_Difference
0     10101614  2.650000 

## Generate Excel Output

In [ ]:

output_path = r'' # Update the output directory path here

with pd.ExcelWriter(output_path) as writer:
    final_output_df.to_excel(writer, sheet_name='GPA_Analysis', index=False)
    df_records.to_excel(writer, sheet_name='Student_Records', index=False)

print(f"\nSuccess! File correctly saved to {output_path} with both sheets.")


Success! File correctly saved to C:\Users\Vinay Attri\Documents\Linkedin\SCI_Performance_Analysis.xlsx with both sheets.
